# Consultas
Estas consultas esta basadas en las necesidades analiticas planteadas en el informe numero 1.

## Consulta 1: ¿Qué local genera más ventas?

In [0]:
%sql
SELECT 
    t.id_tienda,
    t.direccion_tienda AS local,
    t.ciudad,
    SUM(f.monto_final) AS total_ventas
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_tienda t 
    ON f.id_tienda = t.id_tienda
GROUP BY 
    t.id_tienda, 
    t.direccion_tienda, 
    t.ciudad
ORDER BY 
    total_ventas DESC;

id_tienda,local,ciudad,total_ventas
3,Av Javierra Carrera 045,Temuco,1.513309055E8
2,Hayden 102,Temuco,1.44332679E8
1,Pedro Salinas 0295,Temuco,1.00389348E8


Databricks visualization. Run in Databricks to view.

## Consulta 2: ¿Cuáles son los productos más vendidos cada mes?

In [0]:
%sql
WITH ventas_mensuales AS (
    SELECT 
        CASE t.mes
        WHEN 1 THEN '01-Enero'
        WHEN 2 THEN '02-Febrero'
        WHEN 3 THEN '03-Marzo'
        WHEN 4 THEN '04-Abril'
        WHEN 5 THEN '05-Mayo'
        WHEN 6 THEN '06-Junio'
        WHEN 7 THEN '07-Julio'
        WHEN 8 THEN '08-Agosto'
        WHEN 9 THEN '09-Septiembre'
        WHEN 10 THEN '10-Octubre'
        WHEN 11 THEN '11-Noviembre'
        WHEN 12 THEN '12-Diciembre'
    END AS mes_texto,
        p.id_producto,
        p.nombre AS producto,
        SUM(f.cantidad) AS unidades_totales,
        SUM(f.monto_final) AS recaudacion_total,
        ROW_NUMBER() OVER (PARTITION BY t.mes ORDER BY SUM(f.cantidad) DESC) AS ranking
    FROM workspace.ferreteria_gold.hechos_ventas f
    INNER JOIN workspace.ferreteria_gold.dim_tiempo t 
        ON f.id_tiempo = t.id_tiempo
    INNER JOIN workspace.ferreteria_gold.dim_producto p 
        ON f.id_producto = p.id_producto
    GROUP BY 
        t.mes, 
        p.id_producto, 
        p.nombre
)
SELECT 
    mes_texto,
    id_producto,
    producto,
    unidades_totales,
    recaudacion_total
FROM ventas_mensuales
WHERE ranking = 1
ORDER BY mes_texto DESC;

mes_texto,id_producto,producto,unidades_totales,recaudacion_total
12-Diciembre,13285,ESPATULA PROFESIONAL 2,37,82720.0
11-Noviembre,11900,BANDEJA PINTURA,45,131516.0
10-Octubre,107,SPRAY BLANCO OPACO,42,146280.0
09-Septiembre,39123,SILICONA NEUTRA UNIVERSAL GRAFITO,32,131520.0
08-Agosto,60101,TALADRO INALAMBRICO 21V + 2 BAT WELLO CLD50221,39,4812912.5
07-Julio,17360,ESCUADRA (100 UNID) 25 X 25,49,682479.0
06-Junio,60275,"ESCUADRA CARPINTERA ALUMINIO WELLOO 10""",46,285352.0
05-Mayo,60295,CORTADOR DE TUBO 32MM WELLOO,36,0.0
04-Abril,1300,SPRAY ALTA TEMPERATURA aluminio,39,233919.0
03-Marzo,107,SPRAY BLANCO OPACO,38,139840.0


Databricks visualization. Run in Databricks to view.

## Consulta 3: ¿Cuáles son las tendencias de venta de los productos según el clima?

In [0]:
%sql
SELECT 
    c.tipo_clima,
    p.id_producto,
    p.nombre AS nombre_producto,
    p.categoria,
    p.subcategoria,
    SUM(f.cantidad) AS unidades_totales,
    SUM(f.monto_final) AS recaudacion_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_clima c 
    ON f.id_clima = c.id_clima
INNER JOIN workspace.ferreteria_gold.dim_producto p 
    ON f.id_producto = p.id_producto
GROUP BY 
    c.tipo_clima, 
    p.id_producto, 
    p.nombre,
    p.categoria,
    p.subcategoria

-- Se eligen los dos productos mas vendidos por cada clima
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.tipo_clima ORDER BY unidades_totales DESC) <= 2
ORDER BY 
    c.tipo_clima ASC, 
    unidades_totales DESC;

tipo_clima,id_producto,nombre_producto,categoria,subcategoria,unidades_totales,recaudacion_total
Despejado,10083,AMARRA NEGRO (50 UN) 30 cm,Organización y Sujeción,Amarras Plásticas,79,198135.0
Despejado,40,SPRAY BLANCO BRILLANTE,Pinturas,Aerosoles,68,239016.0
Llovizna densa,60113,MEZCLADOR 1400W WELLOO,Herramientas Eléctricas,Mezcladoras,14,2004660.0
Llovizna densa,60101,TALADRO INALAMBRICO 21V + 2 BAT WELLO CLD50221,Herramientas Eléctricas,Taladros Inalámbricos,14,1858780.0
Llovizna ligera,10072,"ESCUADRA REPISA (24 UNIDADES) 8 X 10""",Quincallería,Escuadras para Repisa,43,34445.0
Llovizna ligera,60241,"HUINCHA MEDIR WELLO 10,0 m",Instrumentos de Medición,Huinchas de Medir,41,402110.0
Llovizna moderada,6,SPRAY ROJO VIVO,Pinturas,Aerosoles,27,94576.0
Llovizna moderada,190,SPRAY LACA BRILLANTE,Pinturas,Aerosoles,20,73600.0
Lluvia leve,143,SPRAY ANTIOXIDO negro,Pinturas,Aerosoles,28,94760.0
Lluvia leve,60120,PISTOLA PINTURA 550W WELLOO ESG15550,Herramientas Eléctricas,Pistolas para Pintar,23,1472782.0


Databricks visualization. Run in Databricks to view.

## Consulta 4: ¿Cuáles son las tendencias de venta de los productos según la época del año?

In [0]:
%sql
SELECT 
    c.estacion,
    p.id_producto,
    p.nombre AS nombre_producto,
    p.categoria,
    p.subcategoria,
    SUM(f.cantidad) AS unidades_totales,
    SUM(f.monto_final) AS recaudacion_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_clima c 
    ON f.id_clima = c.id_clima
INNER JOIN workspace.ferreteria_gold.dim_producto p 
    ON f.id_producto = p.id_producto
GROUP BY 
    c.estacion, 
    p.id_producto, 
    p.nombre,
    p.categoria,
    p.subcategoria

-- Top 2 productos por estacion
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.estacion ORDER BY unidades_totales DESC) <= 2
ORDER BY 
    c.estacion ASC, 
    unidades_totales DESC;

estacion,id_producto,nombre_producto,categoria,subcategoria,unidades_totales,recaudacion_total
Invierno,17360,ESCUADRA (100 UNID) 25 X 25,Quincallería,Escuadras Metálicas,92,1304100.0
Invierno,60279,"PLANA BOTADORA 8"" WELLOO",Albañilería,Planas y Lengüetas,67,299791.5
Otoño,60275,"ESCUADRA CARPINTERA ALUMINIO WELLOO 10""",Instrumentos de Medición,Escuadras,68,433412.0
Otoño,60241,"HUINCHA MEDIR WELLO 10,0 m",Instrumentos de Medición,Huinchas de Medir,58,547684.0
Primavera,107,SPRAY BLANCO OPACO,Pinturas,Aerosoles,80,271768.0
Primavera,11900,BANDEJA PINTURA,Accesorios para Pintar,Bandejas,67,199276.0
Verano,10071,"ESCUADRA REPISA (24 UNIDADES) 6 X 8""",Quincallería,Escuadras para Repisa,73,46330.5
Verano,60232,"DESTORNILLADOR PALETA WELLO 4""",Herramientas Manuales,Destornilladores,72,172437.0


Databricks visualization. Run in Databricks to view.

## Consulta 5: ¿Cómo varía la frecuencia de clientes a lo largo del día?

In [0]:
%sql
SELECT 
    CONCAT(LPAD(t.hora, 2, '0'), ':00') AS hora_del_dia,
    COUNT(DISTINCT f.id_venta) AS cantidad_compras,
    COUNT(DISTINCT f.id_cliente) AS clientes_registrados_unicos,
    SUM(f.cantidad) AS cantidad_productos_vendidos,
    SUM(f.monto_final) AS monto_ventas_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_tiempo t 
    ON f.id_tiempo = t.id_tiempo
GROUP BY 
    t.hora
ORDER BY 
    t.hora ASC;

hora_del_dia,cantidad_compras,clientes_registrados_unicos,cantidad_productos_vendidos,monto_ventas_total
09:00,59,17,2515,6.41110895E7
10:00,47,16,1966,2.6140622E7
11:00,48,26,1967,3.64808585E7
12:00,49,19,2247,3.3685288E7
13:00,54,19,2221,3.38425435E7
14:00,41,15,1844,3.62597055E7
15:00,37,13,1619,3.80030395E7
16:00,53,17,2239,4.06598955E7
17:00,57,22,2600,4.39587425E7
18:00,55,23,2303,4.2911148E7


Databricks visualization. Run in Databricks to view.

## Consulta 6: Como varia la frecuencia de los clientes por año y mes

In [0]:
%sql
SELECT 
    t.anio,
    CASE t.mes
        WHEN 1 THEN '01-Enero'
        WHEN 2 THEN '02-Febrero'
        WHEN 3 THEN '03-Marzo'
        WHEN 4 THEN '04-Abril'
        WHEN 5 THEN '05-Mayo'
        WHEN 6 THEN '06-Junio'
        WHEN 7 THEN '07-Julio'
        WHEN 8 THEN '08-Agosto'
        WHEN 9 THEN '09-Septiembre'
        WHEN 10 THEN '10-Octubre'
        WHEN 11 THEN '11-Noviembre'
        WHEN 12 THEN '12-Diciembre'
    END AS mes,
    COUNT(DISTINCT f.id_venta) AS cantidad_compras,
    -- Aquí sumamos los clientes reales únicos + todas las veces que apareció el id 0
    (COUNT(DISTINCT CASE WHEN f.id_cliente != 0 THEN f.id_cliente END) + 
     SUM(CASE WHEN f.id_cliente = 0 THEN 1 ELSE 0 END)) AS clientes_registrados_unicos,
    SUM(f.cantidad) AS cantidad_productos_vendidos,
    SUM(f.monto_final) AS monto_ventas_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_tiempo t 
    ON f.id_tiempo = t.id_tiempo
GROUP BY 
    t.anio,
    t.mes
ORDER BY 
    t.anio DESC, 
    t.mes DESC;

anio,mes,cantidad_compras,clientes_registrados_unicos,cantidad_productos_vendidos,monto_ventas_total
2025,12-Diciembre,7,24,339,4831542.0
2025,11-Noviembre,6,25,230,3593773.0
2025,10-Octubre,7,18,168,5055474.0
2025,09-Septiembre,11,35,530,7433854.5
2025,08-Agosto,7,44,439,7816777.5
2025,07-Julio,7,25,213,1333424.0
2025,06-Junio,6,16,231,1400066.0
2025,05-Mayo,7,33,302,5065896.0
2025,04-Abril,8,32,361,4104291.5
2025,03-Marzo,5,28,223,1659891.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Consulta 7: ¿Que dia de la semana se compra mas y que dia se compra menos?
Se ordena segun el monto total vendido

In [0]:
%sql
SELECT 
    t.dia_semana,
    COUNT(DISTINCT f.id_venta) AS cantidad_de_compras,
    SUM(f.cantidad) AS cantidad_productos_vendidos,
    SUM(f.monto_final) AS monto_total_ventas
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_tiempo t 
    ON f.id_tiempo = t.id_tiempo
GROUP BY 
    t.dia_semana
ORDER BY 
    cantidad_de_compras DESC;

dia_semana,cantidad_de_compras,cantidad_productos_vendidos,monto_total_ventas
Miércoles,86,3466,7.01286625E7
Lunes,78,3466,8.5219916E7
Jueves,76,3136,4.93225675E7
Martes,69,3092,5.50309745E7
Sábado,67,2823,4.38745535E7
Viernes,66,2526,4.9606853E7
Domingo,58,3012,4.28694055E7


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Consulta 8: Top 10 productos mas vendidos segun cantidad

In [0]:
%sql
SELECT 
    p.id_producto,
    p.nombre AS producto,
    p.categoria,
    SUM(f.cantidad) AS total_unidades_vendidas,
    SUM(f.monto_final) AS monto_total_ventas
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_producto p 
    ON f.id_producto = p.id_producto
GROUP BY 
    p.id_producto, 
    p.nombre,
    p.categoria
ORDER BY 
    total_unidades_vendidas DESC
LIMIT 10;

id_producto,producto,categoria,total_unidades_vendidas,monto_total_ventas
17360,ESCUADRA (100 UNID) 25 X 25,Quincallería,182,2511841.5
107,SPRAY BLANCO OPACO,Pinturas,160,557704.0
23,SPRAY BERMELLON,Pinturas,155,540408.0
10601,CORCHETE WELLOO 6 mm,Fijaciones,153,133497.0
60202,"ALICATE CORTANTE 7"" WELLO",Herramientas Manuales,147,1103048.0
10071,"ESCUADRA REPISA (24 UNIDADES) 6 X 8""",Quincallería,142,89646.0
60241,"HUINCHA MEDIR WELLO 10,0 m",Instrumentos de Medición,141,1322382.0
10266,"PESTILLOS 2""",Cerrajería,131,190200.0
10605,CORCHETE WELLOO 14 mm,Fijaciones,131,202620.0
60279,"PLANA BOTADORA 8"" WELLOO",Albañilería,131,594637.5


Databricks visualization. Run in Databricks to view.

## Consulta 9: Top 10 productos que generan mas ganancias

In [0]:
%sql
SELECT 
    p.id_producto,
    p.nombre AS nombre_producto,
    p.categoria,
    SUM(f.cantidad) AS total_unidades_vendidas,
    SUM(f.monto_final) AS recaudacion_total,
    SUM(f.monto_final - (f.cantidad * f.costo_unitario)) AS ganancia_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_producto p 
    ON f.id_producto = p.id_producto
GROUP BY 
    p.id_producto, 
    p.nombre,
    p.categoria
ORDER BY 
    ganancia_total DESC
LIMIT 10;

id_producto,nombre_producto,categoria,total_unidades_vendidas,recaudacion_total,ganancia_total
60115,COMPRESOR DE AIRE 50L (LIBRE DE ACEITE) WELLOO,Herramientas Eléctricas,118,6.6411976E7,3.6801174E7
60291,GATA HIDRAULICA 3TON WELLOO,Automotriz y Taller,92,2.49915175E7,1.40446215E7
60111,LIJADORA DE PARED 850W WELLOO,Herramientas Eléctricas,97,2.258403E7,1.2524451E7
60102,ESMERIL ANG. INALAMB. 20V + 2 BAT WELLO CAG54115,Herramientas Eléctricas,122,1.9708124E7,1.0755886E7
60113,MEZCLADOR 1400W WELLOO,Herramientas Eléctricas,128,1.7726922E7,1.0026186E7
60109,HIDROLAVADORA 1400 W WELLO,Aseo y Limpieza,94,1.46989575E7,8259299.5
60101,TALADRO INALAMBRICO 21V + 2 BAT WELLO CLD50221,Herramientas Eléctricas,99,1.2374164E7,6851251.0
60110,TALADRO DE IMPACTO INHALAMBRICO 21V PCW28908,Herramientas Eléctricas,51,1.0444817E7,5610782.0
60100,TALADRO 12V + 2 BATERIAS WELLO CLD50212,Herramientas Eléctricas,113,8803279.0,4972014.0
60104,ESMERIL ANG. 180MM 2000W WELLO,Herramientas Eléctricas,63,8609031.0,4829850.0


Databricks visualization. Run in Databricks to view.

## Consulta 10: Productos mas vendidos por categoria
Segun cantidad vendida

In [0]:
%sql
SELECT 
    p.categoria,
    p.id_producto,
    p.nombre AS producto,
    SUM(f.cantidad) AS unidades_totales,
    SUM(f.monto_final) AS recaudacion_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_producto p 
    ON f.id_producto = p.id_producto
GROUP BY 
    p.categoria, 
    p.id_producto, 
    p.nombre

QUALIFY ROW_NUMBER() OVER (PARTITION BY p.categoria ORDER BY unidades_totales DESC) = 1
ORDER BY 
    unidades_totales DESC;

categoria,id_producto,producto,unidades_totales,recaudacion_total
Quincallería,17360,ESCUADRA (100 UNID) 25 X 25,182,2511841.5
Pinturas,107,SPRAY BLANCO OPACO,160,557704.0
Fijaciones,10601,CORCHETE WELLOO 6 mm,153,133497.0
Herramientas Manuales,60202,"ALICATE CORTANTE 7"" WELLO",147,1103048.0
Instrumentos de Medición,60241,"HUINCHA MEDIR WELLO 10,0 m",141,1322382.0
Albañilería,60279,"PLANA BOTADORA 8"" WELLOO",131,594637.5
Cerrajería,10266,"PESTILLOS 2""",131,190200.0
Herramientas Eléctricas,60113,MEZCLADOR 1400W WELLOO,128,1.7726922E7
Accesorios para Pintar,10153,RODILLO CHIPORRO AZUL PROFESIONAL 18 CM,124,400633.0
Adhesivos y Sellantes,30181,POLIURETANO PATE 600ML negro,119,1387530.0


Databricks visualization. Run in Databricks to view.

## Consulta 11: Ventas por membresia 

In [0]:
%sql
SELECT 
    CASE 
        WHEN f.id_cliente = 0 THEN 'Sin membresía'
        ELSE COALESCE(c.membresia, 'Sin membresía')
    END AS tipo_membresia,
    COUNT(DISTINCT f.id_venta) AS cantidad_transacciones,
    SUM(f.cantidad) AS unidades_totales_vendidas,
    SUM(f.monto_final) AS monto_venta_total,
    ROUND(SUM(f.monto_final) / COUNT(DISTINCT f.id_venta), 0) AS valor_promedio_venta
FROM workspace.ferreteria_gold.hechos_ventas f
LEFT JOIN workspace.ferreteria_gold.dim_cliente c 
    ON f.id_cliente = c.id_cliente
GROUP BY 
    CASE 
        WHEN f.id_cliente = 0 THEN 'Sin membresía'
        ELSE COALESCE(c.membresia, 'Sin membresía')
    END
ORDER BY 
    monto_venta_total DESC;

tipo_membresia,cantidad_transacciones,unidades_totales_vendidas,monto_venta_total,valor_promedio_venta
Sin membresía,307,12644,2.3781844E8,774653.0
basica,101,4599,9.2573361E7,916568.0
pro,92,4278,6.56611315E7,713708.0


Databricks visualization. Run in Databricks to view.

## Consulta 12: Cómo varía la frecuencia de los clientes por estación

In [0]:
%sql
SELECT 
    c.estacion,
    COUNT(DISTINCT f.id_venta) AS cantidad_compras,
    COUNT(DISTINCT f.id_cliente) AS clientes_registrados_unicos,
    SUM(f.cantidad) AS cantidad_productos_vendidos,
    SUM(f.monto_final) AS monto_ventas_total
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_clima c 
    ON f.id_clima = c.id_clima
GROUP BY 
    c.estacion
ORDER BY 
    clientes_registrados_unicos DESC;

estacion,cantidad_compras,clientes_registrados_unicos,cantidad_productos_vendidos,monto_ventas_total
Verano,133,41,5528,1.08537232E8
Otoño,125,39,5324,9.10482485E7
Invierno,138,38,6174,1.01685247E8
Primavera,104,34,4495,9.4782205E7


Databricks visualization. Run in Databricks to view.

## Consulta 13: Ventas por año

In [0]:
%sql
SELECT 
    t.anio,
    CASE t.mes
        WHEN 1 THEN '01-Enero'
        WHEN 2 THEN '02-Febrero'
        WHEN 3 THEN '03-Marzo'
        WHEN 4 THEN '04-Abril'
        WHEN 5 THEN '05-Mayo'
        WHEN 6 THEN '06-Junio'
        WHEN 7 THEN '07-Julio'
        WHEN 8 THEN '08-Agosto'
        WHEN 9 THEN '09-Septiembre'
        WHEN 10 THEN '10-Octubre'
        WHEN 11 THEN '11-Noviembre'
        WHEN 12 THEN '12-Diciembre'
    END AS mes,
    COUNT(DISTINCT f.id_venta) AS cantidad_compras,
    COUNT(DISTINCT f.id_cliente) AS clientes_registrados_unicos,
    SUM(f.cantidad) AS cantidad_productos_vendidos,
    COUNT(DISTINCT f.id_producto) AS cantidad_productos_diferentes,
    SUM(f.monto_final) AS monto_ventas_total,
    ROUND(SUM(f.monto_final) / COUNT(DISTINCT f.id_venta), 0) AS valor_venta_promedio
FROM workspace.ferreteria_gold.hechos_ventas f
INNER JOIN workspace.ferreteria_gold.dim_tiempo t 
    ON f.id_tiempo = t.id_tiempo
GROUP BY 
    t.anio,
    t.mes   
ORDER BY 
    t.anio ASC;

anio,mes,cantidad_compras,clientes_registrados_unicos,cantidad_productos_vendidos,cantidad_productos_diferentes,monto_ventas_total,valor_venta_promedio
2020,02-Febrero,11,9,539,52,4388769.5,398979.0
2020,07-Julio,9,3,375,46,4829494.5,536611.0
2020,09-Septiembre,5,3,225,27,3426544.0,685309.0
2020,03-Marzo,12,7,496,54,9178444.0,764870.0
2020,05-Mayo,7,3,244,33,5286892.5,755270.0
2020,11-Noviembre,3,2,114,17,1096501.0,365500.0
2020,10-Octubre,5,3,249,26,3388532.0,677706.0
2020,12-Diciembre,5,2,256,28,2395805.0,479161.0
2020,08-Agosto,6,3,154,18,2042039.5,340340.0
2020,04-Abril,8,4,361,39,8180539.0,1022567.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.